[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/06_pandas_intro/06_4_Data_Cleaning.ipynb)

# 06.4: Data Cleaning

Before you can trust any analysis, you need to ask: is the data actually correct? Real datasets have problems. A passenger's age might not have been recorded. A numeric column might have loaded as text. The same sex might be spelled "Male", "male", and "MALE" in three different rows. The same passenger might appear twice. Each of these problems will silently corrupt your analysis if you do not address it first.

This notebook covers the most common cleaning operations: detecting and handling missing values, fixing data types, removing duplicates, and standardizing string columns. Let's load the data.

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
raw = pd.read_csv(url)
raw.columns = raw.columns.str.lower()
raw = raw.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
print(raw.shape)
raw.head()

(887, 8)


,survived,pclass,name,sex,age,sibsp,parch,fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


887 rows, 8 columns, all values present. This version of the Titanic data has been pre-cleaned. That is unusual, and many real datasets are not this tidy. To make the cleaning tools meaningful, we will introduce realistic gaps into the age column (about 20% missing, matching what other Titanic sources show), work through how to handle them, then build a complete cleaning pipeline at the end.

## 1. Checking for Missing Values

The first question on any new dataset: *what's missing?*

This Titanic source is already clean, with no missing values. That's unusual. Many other versions of this dataset have about 20% of ages missing (not all ships recorded every passenger's age). To make the cleaning tools meaningful, we'll introduce realistic gaps.

In [2]:
# This version is complete; let's verify
raw.isnull().sum()

survived    0
pclass      0
name        0
sex         0
age         0
sibsp       0
parch       0
fare        0
dtype: int64

To demonstrate missing-value tools, we'll introduce gaps at random into the age column. The seed ensures the same rows are selected every time this notebook runs.

Introducing the gaps at random is deliberately the easy case. When values are missing completely at random, filling them with a summary like the median barely shifts the column, so the techniques below are safe to apply without much worry. Real missingness is rarely so well behaved. On the actual Titanic, ages were probably not missing at random: records were spottier for lower-fare, lower-class passengers, so the gaps correlate with class. Keep that contrast in mind, because it is exactly what the next section asks you to weigh when you decide whether to drop or fill.

In [3]:
# Introduce ~20% missing ages, as seen in other Titanic sources
rng = np.random.default_rng(42)
df = raw.copy()
missing_idx = rng.choice(df.index, size=int(0.2 * len(df)), replace=False)
df.loc[missing_idx, "age"] = np.nan

# Now check
missing_counts = df.isnull().sum()
missing_pct    = (missing_counts / len(df) * 100).round(1)
pd.DataFrame({"missing": missing_counts, "pct": missing_pct})

,missing,pct
survived,0,0.0
pclass,0,0.0
name,0,0.0
sex,0,0.0
age,177,20.0
sibsp,0,0.0
parch,0,0.0
fare,0,0.0


177 of 887 ages are now missing. Which specific rows are affected? `.isnull().any(axis=1)` marks every row that has at least one NaN in any column.

In [4]:
# Which rows are affected? Use any(axis=1): does any column in this row have a NaN?
rows_with_missing = df[df.isnull().any(axis=1)]
print(f"Rows with at least one missing value: {len(rows_with_missing)}")
rows_with_missing[["name", "age", "survived"]].head()

Rows with at least one missing value: 177


,name,age,survived
5,Mr. James Moran,NaN,0
19,Mrs. Fatima Masselmani,NaN,1
26,Mr. Farred Chehab Emir,NaN,0
29,Mr. Lalio Todoroff,NaN,0
30,Don. Manuel E Uruchurtu,NaN,0


## 2. Deciding What to Do About Missing Data

177 rows have a missing age, every one of them missing only age, since that is the only column we modified. Once you know what is missing, you have two choices: **drop** the affected rows, or **fill** them with something reasonable.

Which you choose depends on two things. First, how much data would you lose? Dropping 20% of rows is significant and may bias your results if the missing values are not random. Second, why is the data missing? If ages were simply not recorded for some passengers, filling with the median is defensible. If the missingness is systematic (for instance, only the poorest passengers lacked records), a blanket fill can introduce bias.

### Dropping with `.dropna()`

In [5]:
df_dropped = df.dropna()
print(f"Original: {len(df)} rows")
print(f"After dropping any row with missing values: {len(df_dropped)} rows")
print(f"Lost: {len(df) - len(df_dropped)} rows ({(len(df)-len(df_dropped))/len(df):.0%})")

Original: 887 rows
After dropping any row with missing values: 710 rows
Lost: 177 rows (20%)


Plain `.dropna()` removes any row that has a missing value in *any* column. If other columns also had gaps, those rows would be dropped too. The `subset=` argument limits the check to specific columns, preserving rows that are only missing in other columns. Here both calls give 710 rows because age is the only column with missing values, but in a dataset with gaps in several columns, they would diverge.

In [6]:
# More surgical: drop only rows where 'age' is missing
# Other columns (if they had missing values) would still be kept
df_dropped_age = df.dropna(subset=["age"])
print(f"Rows after dropping only age-missing rows: {len(df_dropped_age)}")

Rows after dropping only age-missing rows: 710


### Filling with `.fillna()`

Often you'd rather fill than lose data. The most common approach for a numeric column like age is to fill with the **median**, the middle value, which is more robust to outliers than the mean.

In [7]:
median_age = df["age"].median()
print(f"Median age (computed on non-missing rows): {median_age}")

df_filled = df.copy()
df_filled["age"] = df_filled["age"].fillna(median_age)
print(f"Missing after fill: {df_filled['age'].isnull().sum()}")

Median age (computed on non-missing rows): 28.0
Missing after fill: 0


When multiple columns have missing values, pass a dictionary to `.fillna()` so each column gets an appropriate fill value.

In [8]:
# You can fill multiple columns at once with a dict: each column gets its own value
df_filled2 = df.fillna({
    "age": df["age"].median(),    # numeric: use median
    # add more columns here if they had missing values
})
df_filled2.isnull().sum()

survived    0
pclass      0
name        0
sex         0
age         0
sibsp       0
parch       0
fare        0
dtype: int64

> **Machine learning warning:** When you later split data into training and test sets, always compute the fill value (median, mean, mode) from the *training set only*, then apply that same value to the test set. Computing from the full dataset lets information from the test set leak into your model, a subtle but significant bug.

## 3. Duplicate Rows

Duplicates are less common than missing values but worth checking. They can appear when data from multiple sources is combined, or when a data entry process allows re-submission.

In [9]:
print("Duplicates in the original dataset:", raw.duplicated().sum())

Duplicates in the original dataset: 0


The original dataset has zero duplicates. To demonstrate the tools, we'll build a small sample with deliberate repeats using `.iloc[]` to select the same rows more than once.

In [10]:
# Build a small sample with deliberate repeats
# .iloc[] with a list of positions; rows 0, 1, 2 appear twice
df_dupes = raw.iloc[[0, 1, 2, 3, 4, 0, 1, 2]].reset_index(drop=True)
print(f"Rows including duplicates: {len(df_dupes)}")
print(f"Duplicate rows detected:   {df_dupes.duplicated().sum()}")

# Show which rows are duplicates
df_dupes[df_dupes.duplicated(keep=False)][["name", "age", "pclass"]]

Rows including duplicates: 8
Duplicate rows detected:   3


,name,age,pclass
0,Mr. Owen Harris Braund,22.0,3
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,38.0,1
2,Miss. Laina Heikkinen,26.0,3
5,Mr. Owen Harris Braund,22.0,3
6,Mrs. John Bradley (Florence Briggs Thayer) Cum...,38.0,1
7,Miss. Laina Heikkinen,26.0,3


`.duplicated()` marks each row that is an exact repeat of a previous row. `keep=False` marks both the original and the copy, so you can see all the affected rows. Rows 5, 6, and 7 are exact copies of rows 0, 1, and 2. Now remove them.

In [11]:
df_clean = df_dupes.drop_duplicates()
print(f"Rows after removing duplicates: {len(df_clean)}")

Rows after removing duplicates: 5


Back to 5 rows: the three duplicates are gone. `.drop_duplicates()` keeps the first occurrence of each duplicate by default.

## 4. Renaming Columns

Column names from raw data are often inconsistent, long, or hard to type. Rename them early so the rest of your code is cleaner.

In [12]:
# .rename() takes a dict: {old_name: new_name}
df_renamed = raw.rename(columns={
    "sibsp": "siblings_spouses",
    "parch": "parents_children",
    "pclass": "passenger_class",
})
print(df_renamed.columns.tolist())

['survived', 'passenger_class', 'name', 'sex', 'age', 'siblings_spouses', 'parents_children', 'fare']


The column names now read more clearly and will be easier to type in subsequent code.

## 5. Fixing Data Types

Some columns are stored as numbers but really stand for categories. `pclass` holds 1, 2, or 3, but those are labels for first, second, and third class, not quantities. The "average" `pclass` of 2.3 that `.describe()` reported earlier is a meaningless number; there is no such thing as class 2.3. Telling pandas that `pclass` is a category records that it is a label, not a measurement, and `sex` is categorical for the same reason.

`survived` is also categorical in spirit: its 0 and 1 are codes for two outcomes, did not survive and survived. But we deliberately leave it as an integer, because that 0/1 coding is exactly what lets `df.groupby(...)["survived"].mean()` return the survival rate. Averaging the codes is meaningful here precisely because they are 0 and 1, so keeping the column numeric is the more useful choice.

In [13]:
print("Before:")
print(raw.dtypes)
print()
df_typed = raw.copy()
df_typed["pclass"]   = df_typed["pclass"].astype("category")
df_typed["sex"]      = df_typed["sex"].astype("category")

print("After:")
print(df_typed.dtypes)

Before:
survived      int64
pclass        int64
name            str
sex             str
age         float64
sibsp         int64
parch         int64
fare        float64
dtype: object

After:
survived    category
pclass      category
name             str
sex         category
age          float64
sibsp          int64
parch          int64
fare         float64
dtype: object


After converting, `pclass` and `sex` are now `category` type, while `survived` stays an integer so its mean still gives the survival rate; the other numeric and string columns are unchanged. A related problem is a column that *should* be numeric but was read as text because some cells contained non-numeric entries. `pd.to_numeric()` handles that.

In [14]:
# pd.to_numeric() handles columns that should be numeric but contain bad strings
# errors='coerce' converts unparseable values to NaN rather than crashing
messy_ages = pd.Series(["22", "38", "unknown", "35", "28"])
pd.to_numeric(messy_ages, errors="coerce")

0    22.0
1    38.0
2     NaN
3    35.0
4    28.0
dtype: float64

The string "unknown" became `NaN` because it could not be parsed as a number. `errors='coerce'` is the key: without it, `pd.to_numeric()` would raise an error on the first bad value.

## 6. Cleaning String Columns

In notebook 06.1 you used `.str.extract()` to pull titles out of the `name` column. The same `.str` accessor works on any string column. The most common cleaning need is not extraction but **normalization**: the same value spelled inconsistently because of capitalization, leading spaces, or trailing whitespace. Imagine the `sex` column had been entered by hand:

In [15]:
# A messy sex column from manual data entry
messy = pd.Series(["  Male", "female", "MALE", "Female ", "male"])
cleaned = messy.str.strip().str.lower()
print(cleaned.unique())

<StringArray>
['male', 'female']
Length: 2, dtype: str


After `.strip()` and `.lower()`, five messy variants reduced to two clean values: `'male'` and `'female'`. String normalization is almost always necessary before comparing or grouping by a text column.

## 7. A Complete Cleaning Workflow

The sections above demonstrated each tool in isolation. Here they come together as a function that takes a raw CSV and returns a clean, analysis-ready DataFrame.

In [16]:
def prepare_titanic(raw_df):
    df = raw_df.copy()

    # Standardize column names
    df.columns = df.columns.str.lower()
    df = df.rename(columns={
        "siblings/spouses aboard": "sibsp",
        "parents/children aboard": "parch",
    })

    # Fill missing ages with the median if any are missing
    # (many Titanic sources have ~20% of ages missing)
    if df["age"].isnull().any():
        df["age"] = df["age"].fillna(df["age"].median())

    # Extract title; str.extract returns a DataFrame, so take column 0
    df["title"] = df["name"].str.extract(r"^(?:the )?(\w+\.)")[0]
    df = df.drop(columns=["name"])

    # Derive a family-aboard feature
    df["has_family"] = ((df["sibsp"] > 0) | (df["parch"] > 0)).astype(int)

    # Fix data types for categorical columns
    for col in ["pclass", "sex", "title"]:
        df[col] = df[col].astype("category")

    return df


url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
clean = prepare_titanic(pd.read_csv(url))

print(clean.dtypes)
print()
print("Missing values:", clean.isnull().sum().sum())
print()
clean.head()

survived      category
pclass        category
sex           category
age            float64
sibsp            int64
parch            int64
fare           float64
title         category
has_family       int64
dtype: object

Missing values: 0



,survived,pclass,sex,age,sibsp,parch,fare,title,has_family
0,0,3,male,22.0,1,0,7.2500,Mr.,1
1,1,1,female,38.0,1,0,71.2833,Mrs.,1
2,1,3,female,26.0,0,0,7.9250,Miss.,0
3,1,1,female,35.0,1,0,53.1000,Mrs.,1
4,0,3,male,35.0,0,0,8.0500,Mr.,0


The cleaned DataFrame has nine columns: the original numeric columns, a `title` column extracted from names, and a binary `has_family` flag. All categorical columns are typed correctly, and there are no missing values. Because this source is complete, no ages needed filling here; the pipeline fills with the global median only when gaps exist, a sound default when missingness is roughly random and one you would reconsider when it tracks class, as it likely did on the real Titanic. The `name` column has been dropped because the useful information it contained (the title) has been extracted; the rest of the name is too specific to use as a feature.

A function like this is the cleaning step you would run once at the start of a serious analysis. The remaining notebooks in this module reload the raw data and re-derive only the pieces they need, so each one stays self-contained and runnable on its own.

## What's next

You have a cleaned, trustworthy dataset. The next question is comparison: how do different groups of passengers differ from one another? Computing the overall average fare tells you one number about 887 passengers. Computing the average fare separately for each class tells you something far more interesting. Notebook 06.5 introduces GroupBy, the operation that divides the data into groups, computes a summary within each group, and combines the results into a table you can compare.